In [1]:
import sys
sys.path.append('../')

from core import LSTransferTreeBoost
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from utils import * #only needed for xgboost
import matplotlib.pyplot as plt
from baselines import *

import pandas as pd
from sklearn.model_selection import train_test_split

from adapt.instance_based import TrAdaBoostR2
from sklearn.metrics import mean_squared_error, mean_absolute_error

from adapt.instance_based import TrAdaBoostR2

from lineartree import LinearTreeRegressor
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings("ignore")

import forest_config as c

In [2]:
#ablation study for transfertreeboost
for d in c.d_list:
    ablation_transfer_real = pd.DataFrame(columns = ['seed', 'target_column', 'target_instances', 'method',
                                   'v', 'target_tree_size', 'val_rmse', 'val_mae', 'rmse', 'mae'])
    for seed in c.seed_list:
        for train_size in c.train_size_list:
            for target_column in c.target_columns:
            

                #data from Sweden
                data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])
                data_sweden = data_sweden[data_sweden['area_code'] == d]
                print(len(data_sweden))
                data_sweden = data_sweden.sample(1000, random_state=seed) #random sample of 1000 source instances


                #evaluate and train on latvia 
                #data from latvia target
                data_latvia = pd.read_csv(r'../datasets/rs_lettland.csv', index_col=[0])
                data_latvia = data_latvia.rename(columns = {'H_AVERAGE': 'Hgv', 'D_AVERAGE': 'Dgv', 'VOLUME': 'Volume'})
                data_temp, data_test = train_test_split(data_latvia, test_size=0.25, random_state=seed)
                data_train, data_val = train_test_split(data_temp, test_size=0.333, random_state=seed)
                train_size_ = int(len(data_train)*train_size)
                data_train = data_train[0:train_size_]

                #"General" base dataset (to use for transfer)
                X_source_train = np.array(data_sweden[c.predictor_columns])
                y_source_train = np.array(data_sweden[target_column]) #change this to "Dgv" to use diameter as source label!

                #Specific train and test set
                X_target_train = np.array(data_train[c.predictor_columns])
                y_target_train = np.array(data_train[target_column])

                X_target_val = np.array(data_val[c.predictor_columns])
                y_target_val = np.array(data_val[target_column])

                X_target_test = np.array(data_test[c.predictor_columns])
                y_target_test = np.array(data_test[target_column])

                print(len(X_target_train), len(X_target_val), len(X_target_test))

                for config in c.param_grid_tradaboost:
                    n_estimators, lr, tree_size = config  

                    method = f'TradaBoostR2'
                    base_estimator = LinearTreeRegressor(
                        base_estimator=LinearRegression(),
                        max_depth=tree_size,
                        min_samples_leaf=4)
                    model = TrAdaBoostR2(base_estimator,
                                            n_estimators=n_estimators,
                                            lr=lr)
                    model.fit(X_source_train, y_source_train,
                                X_target_train, y_target_train)
                    preds = model.predict(X_target_test)
                    val_preds = model.predict(X_target_val)
                    val_rmse = np.sqrt(
                        mean_squared_error(val_preds, y_target_val))
                    val_mae = mean_absolute_error(val_preds, y_target_val)
                    rmse = np.sqrt(mean_squared_error(
                        preds, y_target_test))
                    mae = mean_absolute_error(preds, y_target_test)
                    ablation_transfer_real = pd.DataFrame(
                        columns=[
                            'seed', 'target_column', 'target_instances',
                            'method', 'n_estimators', 'lr', 'tree_size',
                            'val_rmse', 'val_mae', 'rmse', 'mae'
                        ])
                    ablation_transfer_real.loc[len(
                        ablation_transfer_real)] = [
                            seed, target_column, train_size_, method,
                            n_estimators, lr, tree_size, val_rmse, val_mae,
                            rmse, mae
                        ]
                    ablation_transfer_real.to_csv(f'results/tradaboost_ablation_HGV_rs_{d}.csv') #if using Dgv as source label change HGV to DGV!

2611
95 475 475
Iteration 0 - Error: 0.1948
Iteration 1 - Error: 0.2170
Iteration 2 - Error: 0.2439
Iteration 3 - Error: 0.2768
Iteration 4 - Error: 0.2911
Iteration 5 - Error: 0.2779
Iteration 6 - Error: 0.2678
Iteration 7 - Error: 0.2605
Iteration 8 - Error: 0.2551
Iteration 9 - Error: 0.2513
Iteration 0 - Error: 0.1662


KeyboardInterrupt: 